## 1) Downloads, Imports, etc.

In [ ]:
# !pip install rdkit
# !pip install torch
# !pip install torchani
# !pip install pennylane
# !pip install requests aiohttp
import pennylane as qml

# pip install rdkit-pypi torch torchani (if using ANI)
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## 2) Define a whole bunch of steps to prepare the molecules from SMILES, compute ani energies, etc. etc.


In [4]:
MILES = "O"  # Water
NAME = "water"

NCONF = 150                 # more for larger side chains
RMS_PRUNE = 0.4             # Å
KEEP_MMFF = 50              # keep this many lowest by MMFF energy
USE_ANI = True             # set True to compute ANI-2x energies here

def prepare_mol(smiles):
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)
    return m

def embed_minimize_confs(mol, nconf=NCONF):
    params = AllChem.ETKDGv3()
    params.pruneRmsThresh = -1.0   # we’ll prune ourselves
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=nconf, params=params)
    # MMFF minimize
    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94s')
    e_list = []
    for cid in conf_ids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            e = 1e9
        e_list.append((cid, e))
    return e_list

def prune_by_rmsd(mol, conf_ids, rms_cut=RMS_PRUNE):
    kept = []
    for cid in conf_ids:
        keep = True
        for kc in kept:
            rms = rdMolAlign.GetBestRMS(mol, mol, prbId=cid, refId=kc)
            if rms < rms_cut:
                keep = False
                break
        if keep:
            kept.append(cid)
    return kept

def mmff_rank_and_prune(mol, conf_energy_pairs, keep=KEEP_MMFF):
    conf_energy_pairs = sorted(conf_energy_pairs, key=lambda x: x[1])
    # Take top 'keep' by energy but ensure diversity with RMSD pruning
    ranked = [cid for cid,_ in conf_energy_pairs]
    diverse = prune_by_rmsd(mol, ranked, rms_cut=RMS_PRUNE)
    # keep the best among those diverse; if too many, cap at 'keep'
    diverse_sorted = sorted(diverse, key=lambda cid: dict(conf_energy_pairs)[cid])
    return diverse_sorted[:keep]

def compute_ani_energies(mol, conf_ids, model_name='ani2x'):
    import torch, torchani
    # Load model
    model = (torchani.models.ANI2x() if model_name.lower() == 'ani2x'
             else torchani.models.ANI1ccx())
    device = torch.device('cpu')
    model = model.to(device).eval()

    # Map atomic numbers -> element symbols for TorchANI
    z2sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F', 16:'S', 17:'Cl', 35:'Br', 53:'I'}
    symbols = [z2sym[atom.GetAtomicNum()] for atom in mol.GetAtoms()]

    # TorchANI helper to build species tensor
    species = model.consts.species_to_tensor(symbols).unsqueeze(0).to(device)  # shape (1, natoms)

    energies = {}
    for cid in conf_ids:
        conf = mol.GetConformer(cid)
        coords = [[conf.GetAtomPosition(i).x,
                   conf.GetAtomPosition(i).y,
                   conf.GetAtomPosition(i).z] for i in range(mol.GetNumAtoms())]
        coordinates = torch.tensor([coords], dtype=torch.float32, device=device)  # (1, natoms, 3)
        with torch.no_grad():
            e = model((species, coordinates)).energies.item()  # Hartree
        energies[cid] = e
    return energies  # dict: confId -> Eh


def write_sdf(mol, conf_ids, fields, path):
    w = Chem.SDWriter(path)
    for cid in conf_ids:
        m = Chem.Mol(mol)
        m.SetProp("_Name", f"{NAME}_conf{cid}")
        for k,v in fields.items():
            if cid in v:
                m.SetDoubleProp(k, float(v[cid]))
        w.write(m, confId=cid)
    w.close()




### 3) Test for water

In [6]:
##########################
### Test for h2o)
##########################
mol0 = prepare_mol(MILES)
tauts = [mol0]

print(f"Found {len(tauts)} unique tautomers")
best_overall = None  # (energy_Eh, taut_idx, conf_id)

for i, taut in enumerate(tauts):
    confEs = embed_minimize_confs(taut, NCONF)
    keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)

    aniE = {}
    if USE_ANI:
        aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')

    # Report for this tautomer
    if USE_ANI and aniE:
        cid_min = min(aniE, key=lambda k: aniE[k])
        E_min = aniE[cid_min]   # Hartree
        print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
        if (best_overall is None) or (E_min < best_overall[0]):
            best_overall = (E_min, i, cid_min)
    else:
        # fall back to MMFF (NOT electronic) just so something prints
        mmffE = dict(confEs)
        cid_min = min(keep_ids, key=lambda k: mmffE[k])
        print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")

if USE_ANI and best_overall:
    E, ti, ci = best_overall
    print(f"\nGround-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")

Found 1 unique tautomers


/Users/sanskriti/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torchani/aev.py:16: UserWarning: cuaev not installed
  warnings.warn("cuaev not installed")
/Users/sanskriti/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torchani/__init__.py:55: UserWarning: Dependency not satisfied, torchani.ase will not be available
  warnings.warn("Dependency not satisfied, torchani.ase will not be available")


/Users/sanskriti/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -76.388251401 Eh  (conf 28)

Ground-state estimate (ANI): -76.388251401 Eh  from tautomer 0, conformer 28


### 4) estimate for molecules.

In [ ]:
def estimate_gse_for_molecules(smiles_list, names, out_file="estimations_gse.txt"):
    with open(out_file, "w") as f:
        for SMILES, NAME in zip(smiles_list, names):
            print(f"\nProcessing: {NAME} ({SMILES})")
            mol0 = prepare_mol(SMILES)
            tauts = [mol0]
            best_overall = None  # (energy_Eh, taut_idx, conf_id)
            for i, taut in enumerate(tauts):
                confEs = embed_minimize_confs(taut, NCONF)
                keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)
                aniE = {}
                if USE_ANI:
                    aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')
                if USE_ANI and aniE:
                    cid_min = min(aniE, key=lambda k: aniE[k])
                    E_min = aniE[cid_min]   # Hartree
                    print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
                    if (best_overall is None) or (E_min < best_overall[0]):
                        best_overall = (E_min, i, cid_min)
                else:
                    mmffE = dict(confEs)
                    cid_min = min(keep_ids, key=lambda k: mmffE[k])
                    print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")
            if USE_ANI and best_overall:
                E, ti, ci = best_overall
                result = f"{NAME}\t{SMILES}\t{E:.9f} Eh\tTautomer {ti}\tConformer {ci}\n"
                print(f"Ground-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")
                f.write(result)
            else:
                result = f"{NAME}\t{SMILES}\tNo ANI result\n"
                f.write(result)

# Example usage:
smiles_list = [
    "O",  # Water
    "N[C@@H](C)C(=O)O",         # Alanine
    "CC(C)C(N)C(=O)O",          # Valine
    "CC(C)CC(N)C(=O)O",         # Leucine
    "CC(C)C(N)C(=O)O",          # Isoleucine
    "C(C(C(=O)O)N)S",           # Cysteine
    "NC(CC(=O)O)C(=O)O",        # Asparagine
]
names = ["water", "alanine", "valine", "leucine", "isoleucine", "cysteine", "asparagine"]

estimate_gse_for_molecules(smiles_list, names)


Processing: water (O)
/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -76.388251398 Eh  (conf 88)
Ground-state estimate (ANI): -76.388251398 Eh  from tautomer 0, conformer 88

Processing: alanine (N[C@@H](C)C(=O)O)
/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -323.661033081 Eh  (conf 123)
Ground-state estimate (ANI): -323.661033081 Eh  from tautomer 0, conformer 123

Processing: valine (CC(C)C(N)C(=O)O)
/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -402.271174230 Eh  (conf 38)
Ground-state estimate (ANI): -402.271174230 Eh  from tautomer 0, conformer 38

Processing: leucine (CC(C)CC(N)C(=O)O)
/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -441.575156926 Eh  (conf 109)
Ground-state estimate (ANI): -441.57

## Getting 'true' values from QMPRot Database.
Theirs are calculated using the STO-3G database, so they are quite approximate...
However, this is a good way to get a starting point.

In [ ]:
import pandas as pd

def get_qmprot_energies(names):
    """
    Loads energies for a list of amino acid names from the Pennylane dataset.
    Returns a pandas DataFrame with columns: name, abbreviation, energy.
    """
    records = []
    for name in names:
        data = qml.data.load("other", name=name, attributes=["abbreviation", "energy"])
        if data and hasattr(data[0], "energy"):
            records.append({
                "name": name,
                "abbreviation": getattr(data[0], "abbreviation", ""),
                "energy": data[0].energy
            })
        else:
            records.append({
                "name": name,
                "abbreviation": "",
                "energy": None
            })
    df = pd.DataFrame(records)
    return df

# Example usage:
amino_acids = ["cys", "asn", "ala", "val", "leu", "ile", "ser", "thr", "gly", "pro", "phe", "tyr", "trp", "asp", "glu", "his", "lys", "arg", "met", "gln"]
df_energies = get_qmprot_energies(amino_acids)
print(df_energies)

KeyError: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"

In [ ]:
# ### sample code to just get one
# # from qmprot
# data_cys = qml.data.load("other", name="cys", attributes=["abbreviation", "energy"])
# print("Cysteine energy:", data_cys[0].energy)
# print(dir(data_cys[0]))

Cysteine energy: -710.8573
